# 03_realtime_streaming_architecture
Large Language Models are autoregressive token-by-token generators. Waiting for a complete 500-word response can cause noticeable latency (several seconds), which destroys user experience in chat and conversational UI interfaces.  

In production, streaming tokens via Server-Sent Events (SSE) is standard practice. How streaming works beneath the abstractions, how to handle asynchronous event loops, and how to avoid common networking pitfalls.  

## 1. Core Concepts & Production Metrics
### TTFT (Time to First Token) vs. TPOT (Time Per Output Token):

**TTFT** measures the backend prefill latency (processing prompt tokens into the KV cache) plus initial generation delay.
**TPOT** measures the decoding speed (how fast subsequent tokens are generated and emitted). Streaming optimizes perceived latency by masking TPOT behind immediate rendering.  

### The Reverse Proxy Buffering Trap:
 NGINX, AWS ALB, Cloudflare, or API gateways often enable response buffering by default. If your proxy buffers the upstream response before sending it to the client, streaming silently collapses back into traditional batch loading. Your gateway headers must explicitly disable buffering (X-Accel-Buffering: no).  

## 2. Production Implementation Code (Async Python & FastAPI Pattern)
Using synchronous clients inside an asynchronous web framework like FastAPI will block the event loop, breaking concurrency. Production architectures leverage AsyncOpenAI paired with async generators.

In [ ]:
import asyncio
import os
from openai import AsyncOpenAI

# Initialize the asynchronous client
client = AsyncOpenAI(api_key=os.environ.get("OPENAI_API_KEY"))

async def stream_llm_response(prompt: str):
    """
    Asynchronously streams token deltas from the LLM.
    Designed to feed into an async web framework (e.g., FastAPI StreamingResponse).
    """
    try:
        response_stream = await client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[
                {"role": "system", "content": "You are a concise, high-speed engineering assistant."},
                {"role": "user", "content": prompt}
            ],
            stream=True,
            temperature=0.3
        )
        
        # Iterate over the async stream chunks as they arrive from the provider
        async for chunk in response_stream:
            delta = chunk.choices[0].delta
            if delta and delta.content:
                # Yield data chunk immediately to the client layer
                yield delta.content
                
    except Exception as e:
        yield f"\n[Stream Error Encountered: {str(e)}]"

# Simulation execution runner for local testing
async def main():
    test_prompt = "Write a technical breakdown of how HTTP chunked transfer encoding works."
    print(f"Prompt: {test_prompt}\n\nStreaming Output: ", end="", flush=True)
    
    async for token in stream_llm_response(test_prompt):
        print(token, end="", flush=True)
    print()

if __name__ == "__main__":
    asyncio.run(main())

## 3. Deep-Dive: Architecture & Failure Modes
**Handling Mid-Stream Errors:** What happens if the upstream provider throws a 500 error or a rate limit (429) halfway through a 200-token generation? Because HTTP headers are already sent and status code 200 OK is locked in, you cannot change the status code mid-stream. Your backend must catch the exception inside the stream generator loop and inject an explicit error payload token or a protocol-level termination signal down the wire so the client UI doesn't hang indefinitely.  

**Server-Sent Events (SSE) vs. WebSockets:** 
Use SSE for standard unidirectional text-generation streaming (LLM to client) because it runs natively over HTTP/1.1 or HTTP/2, handles auto-reconnection out of the box, and requires zero custom sub-protocols. 


Choose WebSockets only if you need low-latency, true bi-directional communication streams (e.g., real-time voice-to-voice models or human-in-the-loop interruption features where a user halts text generation midway with a correction).  